In [32]:
import requests
import pandas as pd
import json
import os
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

AERO_API_KEY = os.getenv('AERO_API_KEY')
AIRPORTS = ['EPWA', 'EGLL', 'EDDF']

print("Libraries loaded. AeroDataBox API key secured and ready.")

Libraries loaded. AeroDataBox API key secured and ready.


In [33]:
# PHASE: EXTRACT (AeroDataBox via RapidAPI)


import time

print("Starting AeroDataBox data extraction...\n")

today_str = datetime.now().strftime('%Y-%m-%d')
time_from = f"{today_str}T00:00"
time_to = f"{today_str}T11:59"

headers = {
    "X-RapidAPI-Key": AERO_API_KEY,
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

for airport in AIRPORTS:
    local_file = f"aero_raw_{airport}.json"

    if os.path.exists(local_file):
        print(f"Cache found: {local_file}. Skipping API call.")
    else:
        print(f"Fetching FIDS data for {airport}")

        url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport}/{time_from}/{time_to}"
        querystring = {"withLeg":"true","direction":"Arrival","withCancelled":"false","withCodeshared":"true","withCargo":"false","withPrivate":"false","withLocation":"false"}

        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            json_data = response.json()
            with open(local_file, 'w', encoding='utf-8') as file:
                json.dump(json_data, file, ensure_ascii=False, indent=4)
            print(f"✅ Success! Data saved to {local_file}")

            # API request limit
            time.sleep(2)

        else:
            print(f"❌ Error fetching {airport}: {response.status_code} - {response.text}")

print("\nExtract Phase (AeroDataBox) completed!")

Starting AeroDataBox data extraction...

Cache found: aero_raw_EPWA.json. Skipping API call.
Cache found: aero_raw_EGLL.json. Skipping API call.
Cache found: aero_raw_EDDF.json. Skipping API call.

Extract Phase (AeroDataBox) completed!


In [34]:
# PHASE: TRANSFORM (AeroDataBox Data Cleaning)


import pandas as pd
import json

print("Starting AeroDataBox data transformation in Pandas...\n")
dataframes_list = []
AIRPORTS = ['EPWA', 'EGLL', 'EDDF']

for airport in AIRPORTS:
    local_file = f"aero_raw_{airport}.json"
    try:
        with open(local_file, 'r', encoding='utf-8') as file:
            json_data = json.load(file)

        # 1. Grab ONLY arrivals (AeroDataBox structure)
        flights_list = json_data.get('arrivals', [])

        if not flights_list:
            print(f"No arrivals found for {airport} in the JSON file.")
            continue

        # 2. Flatten JSON
        temp_df = pd.json_normalize(flights_list)

        # 3. Add arrival airport ICAO code manually (since we queried by it)
        temp_df['arrival_airport'] = airport

        dataframes_list.append(temp_df)
    except FileNotFoundError:
        print(f"File {local_file} not found. Run extraction first!")

if dataframes_list:
    raw_flights_df = pd.concat(dataframes_list, ignore_index=True)

    # 4. Map AeroDataBox columns to our Data Warehouse standard
    column_mapping = {
        'number': 'flight_number',
        'status': 'flight_status',
        'airline.name': 'airline',
        'departure.airport.iata': 'departure_airport',
        'arrival_airport': 'arrival_airport',
        'arrival.scheduledTimeUtc': 'scheduled_arrival',
        'arrival.actualTimeUtc': 'actual_arrival'
    }

    existing_columns = [col for col in column_mapping.keys() if col in raw_flights_df.columns]
    clean_flights_df = raw_flights_df[existing_columns].rename(columns=column_mapping)

    # --- BULLETPROOF DATA CLEANING LOGIC ---

    # 5. Filter ONLY flights that have Arrived
    if 'flight_status' in clean_flights_df.columns:
        landed_flights = clean_flights_df[clean_flights_df['flight_status'] == 'Arrived'].copy()
    else:
        landed_flights = clean_flights_df.copy()

    # 6. Standardize datetime formats to UTC
    if 'scheduled_arrival' in landed_flights.columns:
        landed_flights['scheduled_arrival'] = pd.to_datetime(landed_flights['scheduled_arrival'], utc=True)

    if 'actual_arrival' in landed_flights.columns:
        landed_flights['actual_arrival'] = pd.to_datetime(landed_flights['actual_arrival'], utc=True)
        # Drop records without actual arrival time
        landed_flights.dropna(subset=['actual_arrival'], inplace=True)

    # 7. FEATURE ENGINEERING: Calculate Delay in Minutes!
    if 'scheduled_arrival' in landed_flights.columns and 'actual_arrival' in landed_flights.columns:
        # Subtract datetimes and convert to minutes
        landed_flights['delay_minutes'] = (landed_flights['actual_arrival'] - landed_flights['scheduled_arrival']).dt.total_seconds() / 60
        # If delay is negative (arrived early), set it to 0
        landed_flights['delay_minutes'] = landed_flights['delay_minutes'].apply(lambda x: x if x > 0 else 0).astype(int)
    else:
        landed_flights['delay_minutes'] = 0

    print(f"Transformation complete! {len(landed_flights)} landed flights ready for analysis.")

    # Print sample
    cols_to_print = ['flight_number', 'airline', 'arrival_airport', 'delay_minutes']
    print(landed_flights[[c for c in cols_to_print if c in landed_flights.columns]].head().to_string())

    # Ensure it's ready for Cell 4 to save it
else:
    print("No dataframes were created. Check your API response.")

Starting AeroDataBox data transformation in Pandas...

Transformation complete! 2223 landed flights ready for analysis.
  flight_number                airline arrival_airport  delay_minutes
0       W6 1434               Wizz Air            EPWA              0
1       LO 6392  LOT - Polish Airlines            EPWA              0
2       3Z 7149      Smartwings Poland            EPWA              0
3       W6 1540               Wizz Air            EPWA              0
4       QS 746F             SmartWings            EPWA              0


In [35]:
# PHASE: WEATHER EXTRACT (Meteostat Historical) & LOAD TO CSV


from datetime import datetime, timedelta
from meteostat import Point, Hourly
import pandas as pd

print("\nStarting historical weather data extraction...\n")

# Meteostat uses exact geographical coordinates (Latitude, Longitude, Altitude)
# Here are the coordinates for our 3 target airports
airport_locations = {
    'EPWA': Point(52.1657, 20.9671, 110),  # Warsaw Chopin
    'EGLL': Point(51.4700, -0.4543, 25),   # London Heathrow
    'EDDF': Point(50.0333, 8.5705, 111)    # Frankfurt
}

# Define our time window: Last 24 hours
end_time = datetime.now()
start_time = end_time - timedelta(days=1)

weather_frames = []

for airport_code, location in airport_locations.items():
    print(f"Fetching hourly weather data for {airport_code}...")

    # Extract Hourly data from Meteostat
    data = Hourly(location, start_time, end_time)
    data = data.fetch()

    if not data.empty:
        # Meteostat returns data with 'time' as the index. We need it as a column.
        df_airport = data.reset_index()
        # Add our airport code so we can JOIN it later with flights
        df_airport['airport_code'] = airport_code
        weather_frames.append(df_airport)
    else:
        print(f"WARNING: No weather data found for {airport_code} in the given timeframe.")

if weather_frames:
    # Combine all airports into one solid DataFrame
    weather_df = pd.concat(weather_frames, ignore_index=True)

    # Keep only the columns relevant for our Data Warehouse
    # (time, temp, prcp - precipitation/rain, wspd - wind speed, coco - weather condition code)
    columns_to_keep = ['airport_code', 'time', 'temp', 'prcp', 'wspd', 'coco']
    weather_df = weather_df[columns_to_keep]

    # Rename columns for clarity in our SQL database
    weather_df.rename(columns={
        'time': 'weather_timestamp',
        'temp': 'temperature_c',
        'prcp': 'precipitation_mm',
        'wspd': 'wind_speed_kmh',
        'coco': 'condition_code'
    }, inplace=True)

    # Fill any missing weather readings (e.g., missing rain data -> 0 mm)
    weather_df['precipitation_mm'] = weather_df['precipitation_mm'].fillna(0)

    print(f"Weather dataframe created! Total rows: {len(weather_df)}")
    print(weather_df[['airport_code', 'weather_timestamp', 'temperature_c', 'wind_speed_kmh']].head().to_string())

    print("\nSaving cleaned datasets to CSV format...")
    # Assuming 'landed_flights' variable still exists from Cell 3 in memory
    if 'landed_flights' in locals():
        landed_flights.to_csv('clean_flights.csv', index=False, encoding='utf-8')
    weather_df.to_csv('clean_weather.csv', index=False, encoding='utf-8')

    print("Extraction Complete! CSV files are updated and ready for SQL.")
else:
    print("Failed to create weather dataframe.")


Starting historical weather data extraction...

Fetching hourly weather data for EPWA...
Fetching hourly weather data for EGLL...
Fetching hourly weather data for EDDF...
Weather dataframe created! Total rows: 72
  airport_code   weather_timestamp  temperature_c  wind_speed_kmh
0         EPWA 2026-06-24 18:00:00           25.5            14.0
1         EPWA 2026-06-24 19:00:00           24.0             6.0
2         EPWA 2026-06-24 20:00:00           21.0             4.0
3         EPWA 2026-06-24 21:00:00           19.0             2.0
4         EPWA 2026-06-24 22:00:00           19.0             2.0

Saving cleaned datasets to CSV format...
Extraction Complete! CSV files are updated and ready for SQL.
